In [1]:
import numpy as np

In [2]:
# Global constants from FIPS 203 §2.3

# Ring dimension
n = 256

# Prime modulus
q = 3329

# Primitive n-th root of unity modulo q
# ζ = 17 satisfies ζ^256 ≡ 1 (mod q)
zeta = 17

# ML-KEM-768 module dimension
k = 3

# Noise parameter for centered binomial distribution
eta = 2

# Deterministic RNG for reproducible experiments

seed = 0
rng = np.random.default_rng(seed)

print(f"Random seed set to {seed}")

Random seed set to 0


In [3]:
# Polynomial representation in R_q = Z_q[X] / (X^n + 1)
# FIPS 203 §2.4.4
#
# f = f0 + f1 X + ... + f255 X^255
#
# Stored as coefficient array:
# (f0, f1, ..., f255) ∈ Z_q^256

def mod_q(x):
    return x % q


def center_mod_q(x):
    # Map coefficient into interval (-q/2, q/2]
    y = x % q
    if y > q // 2:
        y -= q
    return y


def poly_add(a, b):
    # Coordinate-wise addition in Z_q^n
    return (a + b) % q


def poly_sub(a, b):
    return (a - b) % q

In [4]:
# Noise sampling: SamplePolyCBD_η
# FIPS 203 Algorithm 8
#
# Coefficients follow centered binomial distribution:
#
#     X = sum_{i=1..η} b_i − sum_{i=1..η} b'_i
#
# where b_i, b'_i ~ Bernoulli(1/2)
#
# Output polynomial:
#     f ∈ R_q with coefficients in [-η, η]

def sample_cbd(eta=2):

    f = np.zeros(n, dtype=int)

    for i in range(n):
        a = rng.binomial(1, 0.5, eta).sum()
        b = rng.binomial(1, 0.5, eta).sum()
        f[i] = a - b

    return f % q

In [5]:
# Reference multiplication in R_q
#
# Ring definition:
#     R_q = Z_q[X] / (X^n + 1)
#
# Therefore:
#     X^n ≡ -1
#
# This induces negacyclic convolution:
#
#     (f * g)_k = Σ_i f_i g_{k-i}  -  Σ_i f_i g_{k-i+n}
#
# Used only for correctness testing.

def poly_mul_negacyclic(a, b):

    c = np.zeros(n, dtype=int)

    for i in range(n):
        for j in range(n):

            idx = i + j

            if idx < n:
                c[idx] += a[i] * b[j]
            else:
                c[idx - n] -= a[i] * b[j]

    return c % q

In [6]:
# Number Theoretic Transform
#
# Implements Algorithms 9–12 from FIPS 203
#
# Ring:  R_q = Z_q[X] / (X^256 + 1)
# q = 3329
#
# Representation:
#     f ∈ R_q  →  f̂ ∈ T_q
#
# T_q representation stores 128 degree-1 polynomials
# as coefficient pairs:
#
# (g0,0,g0,1,g1,0,g1,1,...,g127,0,g127,1)

# ζ^{BitRev7(i)} mod q  (Appendix A)

zetas = np.array([
1,1729,2580,3289,2642,630,1897,848,
1062,1919,193,797,2786,3260,569,1746,
296,2447,1339,1476,3046,56,2240,1333,
1426,2094,535,2882,2393,2879,1974,821,
289,331,3253,1756,1197,2304,2277,2055,
650,1977,2513,632,2865,33,1320,1915,
2319,1435,807,452,1438,2868,1534,2402,
2647,2617,1481,648,2474,3110,1227,910,
17,2761,583,2649,1637,723,2288,1100,
1409,2662,3281,233,756,2156,3015,3050,
1703,1651,2789,1789,1847,952,1461,2687,
939,2308,2437,2388,733,2337,268,641,
1584,2298,2037,3220,375,2549,2090,1645,
1063,319,2773,757,2099,561,2466,2594,
2804,1092,403,1026,1143,2150,2775,886,
1722,1212,1874,1029,2110,2935,885,2154
], dtype=np.int64)


# ζ^{2BitRev7(i)+1} mod q  (Appendix A)

gammas = np.array([
17,-17,2761,-2761,583,-583,2649,-2649,
1637,-1637,723,-723,2288,-2288,1100,-1100,
1409,-1409,2662,-2662,3281,-3281,233,-233,
756,-756,2156,-2156,3015,-3015,3050,-3050,
1703,-1703,1651,-1651,2789,-2789,1789,-1789,
1847,-1847,952,-952,1461,-1461,2687,-2687,
939,-939,2308,-2308,2437,-2437,2388,-2388,
733,-733,2337,-2337,268,-268,641,-641,
1584,-1584,2298,-2298,2037,-2037,3220,-3220,
375,-375,2549,-2549,2090,-2090,1645,-1645,
1063,-1063,319,-319,2773,-2773,757,-757,
2099,-2099,561,-561,2466,-2466,2594,-2594,
2804,-2804,1092,-1092,403,-403,1026,-1026,
1143,-1143,2150,-2150,2775,-2775,886,-886,
1722,-1722,1212,-1212,1874,-1874,1029,-1029,
2110,-2110,2935,-2935,885,-885,2154,-2154
], dtype=np.int64) % q


# Algorithm 9 — FIPS 203 §4.3
# Forward NTT over R_q = Z_q[X]/(X^256 + 1)
def ntt(f):
    f = f.copy()
    k_ntt = 1
    length = 128
    while length >= 2:
        start = 0
        while start < 256:
            z = zetas[k_ntt]
            k_ntt += 1
            for j in range(start, start + length):
                t = (z * f[j + length]) % q
                f[j + length] = (f[j] - t) % q
                f[j] = (f[j] + t) % q
            start += 2 * length
        length //= 2
    return f

# Algorithm 10 — FIPS 203 §4.3
# Inverse NTT
def intt(fhat):
    f = fhat.copy().astype(np.int64)
    k_ntt = 127
    length = 2
    while length <= 128:
        start = 0
        while start < 256:
            z = zetas[k_ntt]
            k_ntt -= 1
            for j in range(start, start + length):
                t = f[j]
                f[j] = (t + f[j + length]) % q
                f[j + length] = (z * (f[j + length] - t)) % q
            start += 2 * length
        length *= 2
    inv_n = pow(128, -1, q)
    return (f * inv_n) % q


# Algorithm 12
# BaseCaseMultiply

def base_mul(a0, a1, b0, b1, gamma):

    c0 = (a0*b0 + gamma*a1*b1) % q
    c1 = (a0*b1 + a1*b0) % q

    return c0, c1


# Algorithm 11
# MultiplyNTTs

def multiply_ntts(fhat, ghat):

    h = np.zeros(256, dtype=np.int64)

    for i in range(0,256,2):

        gamma = gammas[i//2]

        h[i],h[i+1] = base_mul(
            fhat[i],fhat[i+1],
            ghat[i],ghat[i+1],
            gamma
        )

    return h


# Polynomial multiplication in R_q
#
# f * g = INTT( NTT(f) × NTT(g) )

def poly_mul(a,b):

    ah = ntt(a)
    bh = ntt(b)

    ch = multiply_ntts(ah,bh)

    return intt(ch)

In [7]:
# Module operations in R_q^k and R_q^(k×k)
#
# Implements module linear algebra over the ring
#
#     R_q = Z_q[X] / (X^n + 1)
#
# Matrix-vector product:
#
#     b_i = Σ_j A_ij · s_j
#
# where multiplication is polynomial multiplication in R_q.

def matvec(A, s):

    b = []

    for i in range(k):

        acc = np.zeros(n, dtype=int)

        for j in range(k):

            prod = poly_mul(A[i][j], s[j])
            acc = poly_add(acc, prod)

        b.append(acc)

    return b

In [8]:
# Uniform polynomial sampling
#
# Coefficients sampled uniformly from Z_q

def random_poly():

    return rng.integers(0, q, size=n)


# Uniform matrix A ∈ R_q^(k×k)

def uniform_matrix():

    A = []

    for i in range(k):

        row = []

        for j in range(k):

            row.append(random_poly())

        A.append(row)

    # print("Generated uniform matrix A with shape:", (k, k))

    return A

In [9]:
# Low-rank matrix generation
#
# A = U V
#
# where
#
#     U ∈ R_q^(k×t)
#     V ∈ R_q^(t×k)
#
# guaranteeing
#
#     rank(A) ≤ t

def low_rank_matrix(t=1):

    U = [[random_poly() for _ in range(t)] for _ in range(k)]
    V = [[random_poly() for _ in range(k)] for _ in range(t)]

    A = [[np.zeros(n, dtype=int) for _ in range(k)] for _ in range(k)]

    for i in range(k):
        for j in range(k):

            acc = np.zeros(n, dtype=int)

            for r in range(t):

                acc = poly_add(acc, poly_mul(U[i][r], V[r][j]))

            A[i][j] = acc

    # print(f"Generated low-rank matrix with rank ≤ {t}")

    return A

In [10]:
# Module-circulant matrix
#
# Defined by generator vector
#
#     (a0, a1, ..., a_{k−1})
#
# Matrix rows are cyclic rotations.

def circulant_matrix():

    a = [random_poly() for _ in range(k)]

    A = []

    for i in range(k):

        row = []

        for j in range(k):

            idx = (j - i) % k
            row.append(a[idx])

        A.append(row)

    # print("Generated module-circulant matrix")

    return A

In [11]:
# Module-Toeplitz matrix
#
# Defined by
#
#     A_ij = t_{j-i}
#
# requiring 2k − 1 polynomials.

def toeplitz_matrix():

    t = [random_poly() for _ in range(2*k - 1)]

    A = []

    for i in range(k):

        row = []

        for j in range(k):

            idx = j - i + (k - 1)
            row.append(t[idx])

        A.append(row)

    # print("Generated module-Toeplitz matrix")

    return A

In [12]:
# Sparse matrix generator
#
# A fraction ρ of entries are nonzero polynomials.

def sparse_matrix(rho=0.3):

    A = []

    total = k * k
    nonzero = int(rho * total)

    positions = rng.choice(total, nonzero, replace=False)

    positions = set(positions)

    for i in range(k):

        row = []

        for j in range(k):

            idx = i * k + j

            if idx in positions:
                row.append(random_poly())
            else:
                row.append(np.zeros(n, dtype=int))

        A.append(row)

    # print(f"Generated sparse matrix with density ρ = {rho}")

    return A

In [13]:
# MLWE sample generation
#
# Implements the relation
#
#     b = A s + e
#
# where
#
#     s,e ~ D_η(R_q)^k

def generate_mlwe_sample(A):

    s = [sample_cbd() for _ in range(k)]
    e = [sample_cbd() for _ in range(k)]

    b = matvec(A, s)

    for i in range(k):
        b[i] = poly_add(b[i], e[i])

    # print("Generated MLWE sample")

    return s, e, b

In [14]:
# Random sample generator
#
# Used for distinguishing experiments
#
#     u ← uniform R_q^k

def generate_uniform_sample(A):

    u = [random_poly() for _ in range(k)]

    # print("Generated uniform random sample")

    return u

In [15]:
# Dataset generation for distinguishing experiments
#
# A is fixed for all N samples to isolate structural effect on b.
# label = 1 → MLWE,  label = 0 → uniform
def generate_dataset(N=1000, matrix_type="uniform", t=1, rho=0.3):
    if matrix_type == "uniform":
        A = uniform_matrix()
    elif matrix_type == "lowrank":
        A = low_rank_matrix(t=t)
    elif matrix_type == "circulant":
        A = circulant_matrix()
    elif matrix_type == "toeplitz":
        A = toeplitz_matrix()
    elif matrix_type == "sparse":
        A = sparse_matrix(rho=rho)
    else:
        raise ValueError(f"Unknown matrix type: {matrix_type}")
    dataset = []
    for _ in range(N):
        if rng.random() < 0.5:
            _, _, b = generate_mlwe_sample(A)
            dataset.append((b, 1))
        else:
            b = generate_uniform_sample(A)
            dataset.append((b, 0))

    return A, dataset

In [16]:
a = random_poly()
b = random_poly()

c1 = poly_mul_negacyclic(a, b)
c2 = poly_mul(a, b)

print("Multiplication correct:", np.all(c1 == c2))

Multiplication correct: True


In [17]:
# Verify NTT inverse correctness
#
# tests: INTT(NTT(f)) == f mod q

f = random_poly()

fh = ntt(f)
f2 = intt(fh)

print("roundtrip correct:", np.all(f % q == f2 % q))

if not np.all(f % q == f2 % q):
    diff = np.where((f % q) != (f2 % q))[0]
    print("first mismatch index:", diff[0])
    print("original:", f[diff[0]])
    print("recovered:", f2[diff[0]])

# Verify convolution theorem
#
# NTT(a*b) == NTT(a) × NTT(b)

a = random_poly()
b = random_poly()

lhs = ntt(poly_mul_negacyclic(a, b))
rhs = multiply_ntts(ntt(a), ntt(b))

print("convolution theorem correct:", np.all(lhs % q == rhs % q))

if not np.all(lhs % q == rhs % q):
    idx = np.where(lhs % q != rhs % q)[0][0]
    print("first mismatch index:", idx)
    print("lhs:", lhs[idx])
    print("rhs:", rhs[idx])

# Inspect first NTT stage

f = random_poly()

f_stage = f.copy()

k_stage = 1          # renamed to avoid clobbering global k
length = 128

z = zetas[k_stage]

print("zeta used:", z)

for j in range(length):
    t = (z * f_stage[j + length]) % q
    new_lo = (f_stage[j] + t) % q
    new_hi = (f_stage[j] - t) % q
    if j < 5:
        print("j", j)
        print("a", f_stage[j], "b", f_stage[j+length])
        print("t", t)
        print("new", new_lo, new_hi)
    f_stage[j] = new_lo
    f_stage[j + length] = new_hi

roundtrip correct: True
convolution theorem correct: True
zeta used: 1729
j 0
a 2442 b 718
t 3034
new 2147 2737
j 1
a 389 b 1661
t 2271
new 2660 1447
j 2
a 1969 b 569
t 1746
new 386 223
j 3
a 823 b 661
t 1022
new 1845 3130
j 4
a 1220 b 2373
t 1589
new 2809 2960


In [18]:
for _ in range(100):
    a = random_poly()
    b = random_poly()
    assert np.all(poly_mul(a,b) == poly_mul_negacyclic(a,b))

print("NTT verified on 100 random tests")

NTT verified on 100 random tests


In [19]:
A = circulant_matrix()

s, e, b = generate_mlwe_sample(A)

print("Example polynomial from b[0]:")
print(b[0][:10])   # first 10 coefficients

Example polynomial from b[0]:
[1900 1937 2111 1520 1428 2942 1274 1142 3281 2200]


In [20]:
A, dataset = generate_dataset(N=10, matrix_type="lowrank", t=1)
print(f"dataset size: {len(dataset)}, label distribution: {sum(l for _,l in dataset)} MLWE / {sum(1-l for _,l in dataset)} uniform")

dataset size: 10, label distribution: 4 MLWE / 6 uniform


In [21]:
def coeff_variance(b):
    # Center all kn = 768 coefficients into (-q/2, q/2] then compute variance
    coeffs = np.array([center_mod_q(c) for poly in b for c in poly], dtype=float)
    return np.var(coeffs)

def variance_threshold(A, n_calib=200):
    # Estimate threshold from uniform samples: median of sigma^2(u)
    vars_uniform = []
    for _ in range(n_calib):
        u = generate_uniform_sample(A)
        vars_uniform.append(coeff_variance(u))
    return np.median(vars_uniform)

def variance_distinguisher(b, tau):
    # Classify as MLWE if variance is below threshold
    return 1 if coeff_variance(b) < tau else 0

# Run variance test across all matrix families
configs = [
    ("lowrank",   dict(t=1)),
    ("lowrank",   dict(t=2)),
    ("circulant", dict()),
    ("toeplitz",  dict()),
    ("sparse",    dict(rho=0.3)),
    ("sparse",    dict(rho=0.5)),
    ("sparse",    dict(rho=0.8)),
]

N = 5000
print(f"{'matrix type':<20} {'threshold':>12} {'accuracy':>10}")
print(f"{'':->20} {'':->12} {'':->10}")

for mtype, mkwargs in configs:
    A, dataset = generate_dataset(N=N, matrix_type=mtype, **mkwargs)
    tau = variance_threshold(A)

    correct = 0
    for b, label in dataset:
        pred = variance_distinguisher(b, tau)
        if pred == label:
            correct += 1

    acc = correct / N
    label_str = mtype if not mkwargs else f"{mtype}({', '.join(f'{k}={v}' for k,v in mkwargs.items())})"
    print(f"{label_str:<20} {tau:>12.1f} {acc:>10.4f}")

matrix type             threshold   accuracy
-------------------- ------------ ----------
lowrank(t=1)             919956.8     0.4926
lowrank(t=2)             924443.6     0.5044
circulant                921596.0     0.5040
toeplitz                 925356.1     0.4968
sparse(rho=0.3)          923811.7     0.7400
sparse(rho=0.5)          922785.0     0.7386
sparse(rho=0.8)          924987.2     0.5036


In [22]:
def cohens_d(A, n_samples=500):
    # Effect size between MLWE and uniform variance distributions
    vars_mlwe = []
    vars_unif = []
    for _ in range(n_samples):
        _, _, b = generate_mlwe_sample(A)
        vars_mlwe.append(coeff_variance(b))
        u = generate_uniform_sample(A)
        vars_unif.append(coeff_variance(u))
    
    m1, m2 = np.mean(vars_mlwe), np.mean(vars_unif)
    s1, s2 = np.std(vars_mlwe), np.std(vars_unif)
    
    # Pooled std
    s_pooled = np.sqrt((s1**2 + s2**2) / 2)
    return (m2 - m1) / s_pooled, m1, m2, s1, s2

configs_d = [
    ("lowrank",  dict(t=1)),
    ("lowrank",  dict(t=2)),
    ("circulant", dict()),
    ("toeplitz",  dict()),
    ("sparse",   dict(rho=0.3)),
    ("sparse",   dict(rho=0.5)),
    ("sparse",   dict(rho=0.8)),
]

print(f"{'matrix type':<20} {'cohen d':>10} {'mlwe mean':>12} {'unif mean':>12} {'separation/sigma':>16}")
for mtype, mkwargs in configs_d:
    A, _ = generate_dataset(N=1, matrix_type=mtype, **mkwargs)
    d, m1, m2, s1, s2 = cohens_d(A)
    sep = (m2 - m1) / ((s1 + s2) / 2)
    label_str = mtype if not mkwargs else f"{mtype}({', '.join(f'{k}={v}' for k,v in mkwargs.items())})"
    print(f"{label_str:<20} {d:>10.4f} {m1:>12.1f} {m2:>12.1f} {sep:>16.4f}")

matrix type             cohen d    mlwe mean    unif mean separation/sigma
lowrank(t=1)            -0.1560     925570.5     920923.8          -0.1560
lowrank(t=2)             0.0038     921658.4     921770.8           0.0038
circulant               -0.0283     924998.1     924133.3          -0.0283
toeplitz                 0.0255     920898.1     921680.8           0.0255
sparse(rho=0.3)         10.9325     614989.8     921933.8          10.9938
sparse(rho=0.5)          0.0091     921143.9     921413.9           0.0091
sparse(rho=0.8)         -0.0092     922962.1     922692.2          -0.0092


In [23]:
import time
from scipy.stats import binom

def binomial_ci(correct, n, alpha=0.05):
    # Clopper-Pearson exact confidence interval
    lo = binom.ppf(alpha/2, n, correct/n) / n if correct > 0 else 0.0
    hi = binom.ppf(1 - alpha/2, n, correct/n) / n if correct < n else 1.0
    return lo, hi

N_sweep = 5000
rho_values = np.arange(0.05, 1.0, 0.05)

print(f"{'rho':>6} {'accuracy':>10} {'ci_lo':>8} {'ci_hi':>8} {'cohen_d':>10}")
sweep_results = []

for rho in rho_values:
    rho = round(rho, 2)
    A, dataset = generate_dataset(N=N_sweep, matrix_type="sparse", rho=rho)
    tau = variance_threshold(A)
    
    correct = sum(1 for b, label in dataset if variance_distinguisher(b, tau) == label)
    acc = correct / N_sweep
    lo, hi = binomial_ci(correct, N_sweep)
    
    d, _, _, _, _ = cohens_d(A, n_samples=300)
    
    sweep_results.append((rho, acc, lo, hi, d))
    print(f"{rho:>6.2f} {acc:>10.4f} {lo:>8.4f} {hi:>8.4f} {d:>10.4f}")

   rho   accuracy    ci_lo    ci_hi    cohen_d
  0.05     0.7804   0.7688   0.7918    46.5353
  0.10     0.7672   0.7554   0.7788    42.8675
  0.15     0.7590   0.7470   0.7708    25.0796
  0.20     0.7500   0.7380   0.7620    25.9378
  0.25     0.7684   0.7566   0.7800    11.3748
  0.30     0.7606   0.7488   0.7724    11.6729
  0.35     0.4996   0.4858   0.5134     0.0799
  0.40     0.7438   0.7316   0.7558    11.3436
  0.45     0.7592   0.7474   0.7710    11.0990
  0.50     0.7636   0.7518   0.7754    11.5403
  0.55     0.7684   0.7566   0.7800    11.9008
  0.60     0.7748   0.7632   0.7864    11.3192
  0.65     0.4968   0.4830   0.5106    -0.0193
  0.70     0.5118   0.4980   0.5256     0.0479
  0.75     0.4912   0.4774   0.5050     0.1442
  0.80     0.4928   0.4790   0.5066     0.0640
  0.85     0.4974   0.4836   0.5112    -0.1561
  0.90     0.4958   0.4820   0.5096    -0.0191
  0.95     0.5000   0.4862   0.5138    -0.1080


In [24]:
def time_matvec(A, n_trials=200):
    # Wall-clock time for As, the dominant ML-KEM operation
    s = [sample_cbd() for _ in range(k)]
    times = []
    for _ in range(n_trials):
        t0 = time.perf_counter()
        matvec(A, s)
        times.append(time.perf_counter() - t0)
    return np.mean(times) * 1000, np.std(times) * 1000  # ms

def time_matrix_gen(matrix_type, mkwargs, n_trials=100):
    # Wall-clock time for matrix generation
    times = []
    for _ in range(n_trials):
        t0 = time.perf_counter()
        if matrix_type == "uniform":
            uniform_matrix()
        elif matrix_type == "lowrank":
            low_rank_matrix(**mkwargs)
        elif matrix_type == "circulant":
            circulant_matrix()
        elif matrix_type == "toeplitz":
            toeplitz_matrix()
        elif matrix_type == "sparse":
            sparse_matrix(**mkwargs)
        times.append(time.perf_counter() - t0)
    return np.mean(times) * 1000, np.std(times) * 1000  # ms

timing_configs = [
    ("uniform",   dict()),
    ("lowrank",   dict(t=1)),
    ("lowrank",   dict(t=2)),
    ("circulant", dict()),
    ("toeplitz",  dict()),
    ("sparse",    dict(rho=0.3)),
    ("sparse",    dict(rho=0.5)),
    ("sparse",    dict(rho=0.8)),
]

print(f"{'matrix type':<20} {'gen mean(ms)':>14} {'gen std':>10} {'matvec mean(ms)':>16} {'matvec std':>12} {'speedup':>10}")
baseline_matvec = None

for mtype, mkwargs in timing_configs:
    A, _ = generate_dataset(N=1, matrix_type=mtype, **mkwargs)
    gen_mean, gen_std = time_matrix_gen(mtype, mkwargs)
    mv_mean, mv_std = time_matvec(A)
    
    if baseline_matvec is None:
        baseline_matvec = mv_mean
    
    speedup = baseline_matvec / mv_mean
    label_str = mtype if not mkwargs else f"{mtype}({', '.join(f'{k}={v}' for k,v in mkwargs.items())})"
    print(f"{label_str:<20} {gen_mean:>14.3f} {gen_std:>10.3f} {mv_mean:>16.3f} {mv_std:>12.3f} {speedup:>10.3f}x")

matrix type            gen mean(ms)    gen std  matvec mean(ms)   matvec std    speedup
uniform                       0.048      0.013           10.028        0.884      1.000x
lowrank(t=1)                  8.479      0.669           10.006        0.848      1.002x
lowrank(t=2)                 16.744      1.086           10.021        0.880      1.001x
circulant                     0.013      0.001           10.159        0.928      0.987x
toeplitz                      0.024      0.006           10.002        0.739      1.003x
sparse(rho=0.3)               0.020      0.005           11.120        0.959      0.902x
sparse(rho=0.5)               0.031      0.009           10.716        0.745      0.936x
sparse(rho=0.8)               0.044      0.012           10.312        0.782      0.972x


In [26]:
# Sweep over exact nonzero counts rather than rho
# Force exact nonzero count by modifying sparse_matrix

def sparse_matrix_exact(n_nonzero):
    # Exactly n_nonzero polynomial entries are nonzero
    total = k * k
    positions = rng.choice(total, n_nonzero, replace=False)
    positions = set(positions)
    A = []
    for i in range(k):
        row = []
        for j in range(k):
            idx = i * k + j
            if idx in positions:
                row.append(random_poly())
            else:
                row.append(np.zeros(n, dtype=int))
        A.append(row)
    return A

def generate_dataset_exact_sparse(N=5000, n_nonzero=1):
    A = sparse_matrix_exact(n_nonzero)
    dataset = []
    for _ in range(N):
        if rng.random() < 0.5:
            _, _, b = generate_mlwe_sample(A)
            dataset.append((b, 1))
        else:
            b = generate_uniform_sample(A)
            dataset.append((b, 0))
    return A, dataset

print(f"{'nonzero':>8} {'accuracy':>10} {'ci_lo':>8} {'ci_hi':>8} {'cohen_d':>10} {'mlwe_var':>12} {'unif_var':>12}")

exact_results = []
for nz in range(1, 10):
    A, dataset = generate_dataset_exact_sparse(N=5000, n_nonzero=nz)
    tau = variance_threshold(A)
    
    correct = sum(1 for b, label in dataset if variance_distinguisher(b, tau) == label)
    acc = correct / 5000
    lo, hi = binomial_ci(correct, 5000)
    d, m1, m2, _, _ = cohens_d(A, n_samples=300)
    
    exact_results.append((nz, acc, lo, hi, d, m1, m2))
    print(f"{nz:>8} {acc:>10.4f} {lo:>8.4f} {hi:>8.4f} {d:>10.4f} {m1:>12.1f} {m2:>12.1f}")

 nonzero   accuracy    ci_lo    ci_hi    cohen_d     mlwe_var     unif_var
       1     0.7256   0.7132   0.7380    26.3521     306926.4     923432.6
       2     0.7668   0.7550   0.7784    11.7325     617108.5     922094.3
       3     0.5116   0.4978   0.5254     0.0917     920749.2     923475.7
       4     0.5104   0.4966   0.5242    -0.1250     924506.6     920892.5
       5     0.5044   0.4906   0.5182    -0.0261     923476.5     922694.1
       6     0.5068   0.4930   0.5206    -0.0378     920856.1     919707.9
       7     0.5016   0.4878   0.5154     0.0887     923137.3     925668.6
       8     0.4992   0.4854   0.5130     0.0270     922786.6     923596.2
       9     0.4922   0.4784   0.5060     0.0103     923834.3     924134.1


In [27]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

def logreg_accuracy(dataset, n_train=4000):
    # Feature vector: centered coefficients of b, shape (kn,)
    X = np.array([[center_mod_q(c) for poly in b for c in poly] 
                  for b, _ in dataset], dtype=float)
    y = np.array([label for _, label in dataset])
    
    X_train, X_test = X[:n_train], X[n_train:]
    y_train, y_test = y[:n_train], y[n_train:]
    
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)
    
    clf = LogisticRegression(max_iter=1000)
    clf.fit(X_train, y_train)
    return clf.score(X_test, y_test)

print(f"{'nonzero':>8} {'var_acc':>10} {'logreg_acc':>12}")

for nz, acc, lo, hi, d, m1, m2 in exact_results:
    A, dataset = generate_dataset_exact_sparse(N=5000, n_nonzero=nz)
    lr_acc = logreg_accuracy(dataset)
    print(f"{nz:>8} {acc:>10.4f} {lr_acc:>12.4f}")

 nonzero    var_acc   logreg_acc
       1     0.7256       0.5690
       2     0.7668       0.5010
       3     0.5116       0.5220
       4     0.5104       0.5170
       5     0.5044       0.5120
       6     0.5068       0.5240
       7     0.5016       0.4810
       8     0.4992       0.5160
       9     0.4922       0.5200


In [29]:
configs_all = [
    ("uniform", dict()),
    ("lowrank",   dict(t=1)),
    ("lowrank",   dict(t=2)),
    ("circulant", dict()),
    ("toeplitz",  dict()),
    ("sparse",    dict(rho=0.3)),
    ("sparse",    dict(rho=0.5)),
    ("sparse",    dict(rho=0.8)),
]

print(f"{'matrix type':<20} {'var_acc':>10} {'logreg_acc':>12} {'cohen_d':>10}")

for mtype, mkwargs in configs_all:
    A, dataset = generate_dataset(N=5000, matrix_type=mtype, **mkwargs)
    tau = variance_threshold(A)
    
    correct = sum(1 for b, label in dataset if variance_distinguisher(b, tau) == label)
    var_acc = correct / 5000
    
    lr_acc = logreg_accuracy(dataset)
    d, _, _, _, _ = cohens_d(A, n_samples=300)
    
    label_str = mtype if not mkwargs else f"{mtype}({', '.join(f'{k}={v}' for k,v in mkwargs.items())})"
    print(f"{label_str:<20} {var_acc:>10.4f} {lr_acc:>12.4f} {d:>10.4f}")

matrix type             var_acc   logreg_acc    cohen_d
uniform                  0.5076       0.4940     0.0035
lowrank(t=1)             0.4986       0.5110    -0.0315
lowrank(t=2)             0.5078       0.5060    -0.0349
circulant                0.5098       0.4980     0.0825
toeplitz                 0.4964       0.5030    -0.0105
sparse(rho=0.3)          0.7416       0.5630    23.9731
sparse(rho=0.5)          0.4950       0.4990     0.0825
sparse(rho=0.8)          0.4886       0.4930     0.0560
